In [1]:
# Cell 1: Setup
import os
import polars as pl
import spatial_split as ss

# Regions to process: set here, or via the REGIONS environment variable (comma separated)
# REGIONS = os.environ.get('REGIONS', 'HIM').split(',')
REGIONS = ['HIM', 'PEN', 'DES', 'IGP', 'WGR', 'ICR', 'NEI']
STRESS_YEARS = {'HIM': [2003, 2001]}   # per region; regions not listed get the weakest unused usable year automatically
N_SPREAD = 8                           # evenly spread search years
plans = {}

In [2]:
# Cell 2: Usable years + search years per region (saved to data/study_years/{REGION}.json).
# Edit data/study_years/overrides.json to exclude years or fix the search years by hand, then rerun this cell.
for REGION in REGIONS:
    print(f"\n{'='*60}\n{REGION}\n{'='*60}")
    plans[REGION] = ss.plan_years(REGION, N_SPREAD, STRESS_YEARS.get(REGION))
    print(plans[REGION]['table'].to_pandas().to_string(index=False))


HIM
[HIM] usable 30 years (1993-2025); search 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025]; stress 2003
 year  rows  polys  n_cls  min_cls  usable
 1987     0      0      0        0   False
 1988  1990     21      6        1   False
 1989  4344     52      9        1   False
 1990  4654     66      8        1   False
 1991 13618    169     10        3   False
 1992 31114    285     10        4   False
 1993 42967    385     11        6    True
 1994 40271    365     11        6    True
 1995  6746     87      9        1   False
 1996 35110    341     11        1   False
 1997 14301    182     11        1   False
 1998 42324    382     11        6    True
 1999 36980    374     11        6    True
 2000 43589    403     11        6    True
 2001 37030    378     11        6    True
 2002 41083    378     11        6    True
 2003 35184    382     11        6    True
 2004 41107    386     11        6    True
 2005 41165    397     11        6    True
 2006 42702    401    

In [3]:
# Cell 3: What was chosen
for REGION, plan in plans.items():
    print(f"[{REGION}] usable {len(plan['usable'])}: {plan['usable']}")
    print(f"[{REGION}] search {len(plan['search'])}: {plan['search']} | stress year: {plan['stress']}")

[HIM] usable 30: [1993, 1994, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[HIM] search 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025] | stress year: 2003
[PEN] usable 21: [2000, 2003, 2004, 2006, 2007, 2008, 2009, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[PEN] search 9: [2000, 2003, 2006, 2009, 2014, 2016, 2019, 2022, 2025] | stress year: 2003
[DES] usable 34: [1991, 1992, 1993, 1994, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
[DES] search 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025] | stress year: 2002
[IGP] usable 30: [1992, 1993, 1994, 1996, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2

In [4]:
# Cell 4: Build the polygon-level split on the usable years (gated rows). ~minutes per region.
# force=False builds it only if missing and otherwise reuses it; use True only to deliberately redraw the split.
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.build_block_splits(REGION, force=False)


--- HIM ---

--- PEN ---
[PEN] building split from 21 usable years, quality-gated
[PEN] 2,944 polygons; pixels per polygon (median, p90, p99): 100.0, 790.4, 4266.6; extent km (median, p90, p99): 0.1, 0.7, 3.8
[PEN] WARNING 0 keys hold more than one class and 17 span > 5 km: probably different polygons sharing one area_m2. See the polygon report.
[PEN] 2,944 polygons in 2,524 centroid blocks of 0.01 deg, seed 61. Class report -> PEN_poly_block0.01_s10000_y21_2557d9_class_report.csv

--- DES ---
[DES] building split from 34 usable years, quality-gated
[DES] 558 polygons; pixels per polygon (median, p90, p99): 2300.0, 15048.2, 89270.8; extent km (median, p90, p99): 0.4, 1.5, 7.9
[DES] WARNING 0 keys hold more than one class and 10 span > 5 km: probably different polygons sharing one area_m2. See the polygon report.
[DES] 558 polygons in 544 centroid blocks of 0.01 deg, seed 84. Class report -> DES_poly_block0.01_s10000_y34_c68d15_class_report.csv

--- IGP ---
[IGP] building split from 30

In [5]:
# Cell 5: Check the split per year. Look for years where val/test miss a class, or are tiny.
for REGION in REGIONS:
    s = ss.split_summary(REGION)
    print(f"\n--- {REGION} ---")
    print(s.to_pandas().to_string(index=False))
    bad = s.filter((pl.col('val_missing') != '') | (pl.col('test_missing') != ''))
    print(f"{REGION}: {bad.height} of {s.height} years have a class missing from val or test")


--- HIM ---
 year  train_px  train_polys  val_px  val_polys val_missing  test_px  test_polys test_missing
 1993     37087          309    4366         55                 1514          21             
 1994     34727          290    4139         52                 1405          23             
 1998     36474          305    4354         56                 1496          21             
 1999     31999          301    3888         54                 1093          19             
 2000     37643          321    4451         58                 1495          24             
 2001     31509          302    4187         55                 1334          21             
 2002     35508          302    4156         55                 1419          21             
 2003     30180          304    3747         55                 1257          23             
 2004     35575          307    4200         56                 1332          23             
 2005     35591          316    4161         57

In [6]:
# Cell 6: Polygon sanity check (same polygon -> one class; polygons stay whole in one split)
for REGION in REGIONS:
    print(f"\n--- {REGION} ---")
    ss.polygon_check(REGION, years=ss.study_years(REGION)[:3])


--- HIM ---
year=1993: 45,029 rows, 0 without area_m2, 420 polygons, pixels/polygon median 40, p90 225
year=1994: 44,886 rows, 0 without area_m2, 422 polygons, pixels/polygon median 38, p90 224
year=1998: 44,966 rows, 0 without area_m2, 419 polygons, pixels/polygon median 40, p90 224

year=1993 vs year=1998: 0 shared system:index values (0.0% of year=1993); same lat/lon on 0; same area_m2 on 0
If shared == 0 the two years sample DIFFERENT pixels; if shared is large and lat/lon agree, the same pixels are reused.
polygon extent km (median, p90, p99, max): 0.48, 1.89, 5.52, 7.80;  keys with >1 class: 0

--- PEN ---
year=2000: 58,316 rows, 0 without area_m2, 2,944 polygons, pixels/polygon median 5, p90 38
year=2003: 58,304 rows, 0 without area_m2, 2,944 polygons, pixels/polygon median 5, p90 38
year=2004: 58,315 rows, 0 without area_m2, 2,944 polygons, pixels/polygon median 5, p90 38

year=2000 vs year=2004: 0 shared system:index values (0.0% of year=2000); same lat/lon on 0; same area_m2

In [7]:
# Cell 7: What the later steps will use
for REGION in REGIONS:
    print(f"{REGION}: usable pool (split, 04 ranking, 06 pruning) {len(ss.study_years(REGION))} years | "
          f"search years (07 Optuna) {len(ss.search_years(REGION))}: {ss.search_years(REGION)}")

HIM: usable pool (split, 04 ranking, 06 pruning) 30 years | search years (07 Optuna) 9: [1993, 2000, 2003, 2004, 2008, 2013, 2017, 2021, 2025]
PEN: usable pool (split, 04 ranking, 06 pruning) 21 years | search years (07 Optuna) 9: [2000, 2003, 2006, 2009, 2014, 2016, 2019, 2022, 2025]
DES: usable pool (split, 04 ranking, 06 pruning) 34 years | search years (07 Optuna) 9: [1991, 1997, 2001, 2002, 2006, 2011, 2016, 2020, 2025]
IGP: usable pool (split, 04 ranking, 06 pruning) 30 years | search years (07 Optuna) 9: [1992, 1996, 2000, 2004, 2008, 2013, 2017, 2021, 2025]
WGR: usable pool (split, 04 ranking, 06 pruning) 19 years | search years (07 Optuna) 9: [2000, 2006, 2007, 2012, 2015, 2017, 2020, 2022, 2025]
ICR: usable pool (split, 04 ranking, 06 pruning) 24 years | search years (07 Optuna) 9: [2001, 2004, 2008, 2011, 2012, 2015, 2018, 2022, 2025]
NEI: usable pool (split, 04 ranking, 06 pruning) 13 years | search years (07 Optuna) 9: [2013, 2014, 2015, 2016, 2018, 2020, 2022, 2023, 2025]